## ENEM 2022 — Recortes geográficos,  variáveis derivadas (pós‐pré-processamento)

Objetivo: a partir do dataset tratado (nível candidato), criar variáveis derivadas usadas no dashboard/EDA e gerar recortes geográficos (MG e Caxambu), mantendo tipagem consistente e reduzindo categorias não utilizadas.

Entradas: DADOS_TRATADOS_2022 (Parquet, saída do notebook 00-fbps-preprocessamento_2022).
Saídas:

DADOS_TRATADOS_2022_MEDIAS (Parquet)  — arquivo original com as duas variáveis derivadas adicionadas

DADOS_TRAT_MG_2022 (Parquet) — candidatos com prova em MG + variável regiao

DADOS_TRAT_CAX_2022 (Parquet) — candidatos com prova em Caxambu

(opcional/etapa posterior) tabelas agregadas por município/região (quando você criar/rodar df_agg)

Observação (GitHub): microdados brutos não são versionados; esta etapa opera sobre Parquet já tratado.

In [1]:
import sys
from pathlib import Path

# Permite importar o pacote `src/` a partir do diretório do projeto.
ROOT_PATH = Path().resolve().parents[1]  # notebooks/00_preprocessamento -> projeto
if str(ROOT_PATH) not in sys.path:
    sys.path.append(str(ROOT_PATH))

import re
import numpy as np
import pandas as pd


from src.config import (
    DADOS_TRATADOS_2022, 
    DADOS_TRATADOS_MEDIAS_2022,
    DADOS_TRAT_MG_2022, 
    DADOS_TRAT_CAX_2022
)

from src.preprocessamento.regioes_mg import  atribuir_regiao, MAP_NOME_REGIAO, MAP_REGIOES
from src.preprocessamento.categorias import MAP_SAL_MIN_RENDA_MEDIA




pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:.2f}")




### 2) Leitura do dataset tratado

In [2]:
df = pd.read_parquet( DADOS_TRATADOS_2022)


df.head(3)

,faixa_etaria,sexo,estado_civil,cor_raca,escola,municipio,uf,nota_cn,nota_ch,nota_lc,nota_mt,lingua,nota_redacao,escolaridade_pai,escolaridade_mae,ocup_pai,ocup_mae,n_pessoas_resd,sal_min,emp_domst,gelad,lv_rp,tv,cel,comptdr,indice_consumo
0,36-45,masculino,casado/mora com companheiro,negra,não informada,Brasília,DF,NaN,NaN,NaN,NaN,inglês,NaN,não estudou,não estudou,rural,básico,3,1 a 3,1,1,1,0,0,0,0.11
1,36-45,masculino,casado/mora com companheiro,branca,não informada,Brasília,DF,NaN,NaN,NaN,NaN,inglês,NaN,até fund,até fund,superior,superior,3,acima de 20,5,4,4,4,4,4,1.00
2,20-25,feminino,solteiro,negra,não informada,Presidente Tancredo Neves,BA,421.10,546.00,498.80,565.30,espanhol,760.00,médio,superior,rural,médio/tec,2,1 a 3,0,1,0,0,2,0,0.15


### 3) Variáveis derivadas

Nesta etapa são criadas variáveis auxiliares que facilitam análises exploratórias, agregações e etapas posteriores do pipeline, incluindo o dashboard e a modelagem preditiva. Essas variáveis não fazem parte dos microdados originais do ENEM, mas são derivadas de campos já existentes para melhorar a interpretabilidade e a usabilidade dos dados.

#### renda_media (proxy numérica de sal_min)

A variável sal_min representa a renda familiar mensal em faixas de salários mínimos, sendo portanto uma categoria ordinal. Para permitir análises quantitativas simples — como correlações, cálculos de médias, rankings e agregações — foi criada a variável renda_media, que associa a cada faixa um valor numérico representativo, aproximando o ponto médio de cada intervalo.

Essa transformação não pretende estimar a renda real do candidato, mas apenas fornecer uma proxy numérica consistente para análises estatísticas e comparações entre grupos.

#### nota_media (indicador sintético de desempenho)

Para obter uma medida sintética do desempenho do candidato no ENEM, foi criada a variável nota_media, definida como a média aritmética das cinco notas do exame:

* Ciências da Natureza (nota_cn)
* Ciências Humanas (nota_ch)
* Linguagens (nota_lc)
* Matemática (nota_mt)
* Redação (nota_redacao)

Essa variável fornece uma visão geral do desempenho global do candidato, sendo útil para:

* comparações entre grupos socioeconômicos
* análises regionais
* visualizações no dashboard
* definição de variável alvo em experimentos de modelagem preditiva

Embora simplifique o desempenho em um único indicador, a média preserva a escala das notas individuais e facilita análises agregadas mais diretas.

##### Observação metodológica

As notas do ENEM são apresentadas em uma escala comparável entre áreas, o que permite a construção de uma medida sintética de desempenho para fins analíticos. Assim, nota_media não substitui análises específicas por área do conhecimento, mas funciona como um indicador global adequado ao objetivo deste projeto, que é investigar padrões socioeconômicos associados ao desempenho educacional.

In [3]:
#renda média

df["renda_media"] = (
    df["sal_min"].astype("string").map(MAP_SAL_MIN_RENDA_MEDIA).astype("float32")
)

In [4]:
#nota média
df["nota_media"] = df[
    ["nota_cn", "nota_ch", "nota_lc", "nota_mt", "nota_redacao"]
].mean(axis=1).astype("float32")

### 4) Recortes geográficos e limpeza de categorias

In [5]:

df_caxambu = df.loc[df['municipio'] == 'Caxambu', :].copy()
df_caxambu['municipio'] = df_caxambu['municipio'].cat.remove_unused_categories()
df_caxambu['uf'] = df_caxambu['uf'].cat.remove_unused_categories()
df_caxambu.head(3)

,faixa_etaria,sexo,estado_civil,cor_raca,escola,municipio,uf,nota_cn,nota_ch,nota_lc,nota_mt,lingua,nota_redacao,escolaridade_pai,escolaridade_mae,ocup_pai,ocup_mae,n_pessoas_resd,sal_min,emp_domst,gelad,lv_rp,tv,cel,comptdr,indice_consumo,renda_media,nota_media
10396,20-25,feminino,solteiro,negra,não informada,Caxambu,MG,NaN,NaN,NaN,NaN,espanhol,NaN,médio,até fund,básico,básico,10,1 a 3,0,1,0,1,4,0,0.26,2.00,NaN
11961,26-35,masculino,solteiro,negra,não informada,Caxambu,MG,NaN,506.20,419.00,NaN,inglês,460.00,desconhecida,não estudou,básico,básico,2,até 1,0,1,0,1,1,0,0.08,0.50,461.73
32214,20-25,feminino,não informado,negra,não informada,Caxambu,MG,439.20,470.70,500.00,571.60,inglês,820.00,médio,até fund,manual/tec,básico,2,1 a 3,0,1,1,1,2,1,0.17,2.00,560.30


In [6]:
df_mg = df.loc[df['uf'] == 'MG', :].copy()
df_mg['uf'] = df_mg['uf'].cat.remove_unused_categories()
df_mg.head(3)

,faixa_etaria,sexo,estado_civil,cor_raca,escola,municipio,uf,nota_cn,nota_ch,nota_lc,nota_mt,lingua,nota_redacao,escolaridade_pai,escolaridade_mae,ocup_pai,ocup_mae,n_pessoas_resd,sal_min,emp_domst,gelad,lv_rp,tv,cel,comptdr,indice_consumo,renda_media,nota_media
18,20-25,feminino,solteiro,negra,não informada,Ladainha,MG,481.40,603.60,589.00,695.00,espanhol,860.00,até fund,superior,rural,médio/tec,6,1 a 3,0,1,1,1,3,0,0.24,2.00,645.80
24,20-25,feminino,solteiro,negra,não informada,Poços de Caldas,MG,NaN,NaN,NaN,NaN,espanhol,NaN,até fund,até fund,desconhecida,rural,1,1 a 3,0,1,1,1,1,1,0.18,2.00,NaN
42,20-25,feminino,solteiro,negra,não informada,Contagem,MG,470.20,340.60,492.30,496.60,espanhol,720.00,até fund,até fund,básico,básico,2,1 a 3,0,1,1,1,2,0,0.19,2.00,503.94


### 5) Variável regiao (macro-regiões dentro de MG)


#### 5.1) Atribuição de região e mapeamento dos nomes para versões mais amigáveis



In [7]:
# Aplicar no dataframe
df_mg["regiao"] = df_mg["municipio"].astype("string").apply(atribuir_regiao)

df_mg["regiao"] = df_mg["regiao"].map(MAP_NOME_REGIAO).astype("category")
df_mg.head(3)




,faixa_etaria,sexo,estado_civil,cor_raca,escola,municipio,uf,nota_cn,nota_ch,nota_lc,nota_mt,lingua,nota_redacao,escolaridade_pai,escolaridade_mae,ocup_pai,ocup_mae,n_pessoas_resd,sal_min,emp_domst,gelad,lv_rp,tv,cel,comptdr,indice_consumo,renda_media,nota_media,regiao
18,20-25,feminino,solteiro,negra,não informada,Ladainha,MG,481.40,603.60,589.00,695.00,espanhol,860.00,até fund,superior,rural,médio/tec,6,1 a 3,0,1,1,1,3,0,0.24,2.00,645.80,Vale do Mucuri
24,20-25,feminino,solteiro,negra,não informada,Poços de Caldas,MG,NaN,NaN,NaN,NaN,espanhol,NaN,até fund,até fund,desconhecida,rural,1,1 a 3,0,1,1,1,1,1,0.18,2.00,NaN,Sul de Minas
42,20-25,feminino,solteiro,negra,não informada,Contagem,MG,470.20,340.60,492.30,496.60,espanhol,720.00,até fund,até fund,básico,básico,2,1 a 3,0,1,1,1,2,0,0.19,2.00,503.94,Metrop. de Belo Horizonte


#### 5.2) Diagnóstico: municípios não classificados

In [8]:
todas = set()
for cidades in MAP_REGIOES.values():
    todas.update(cidades)

municipios_df = df_mg["municipio"].astype("string").str.strip().unique()
nao_classificados = [m for m in municipios_df if m not in todas]

print("Municípios únicos em MG:", len(municipios_df))
print("Não classificados:", len(nao_classificados))
# se quiser, mostre só os primeiros 30 para não poluir o notebook
print("Exemplos:", nao_classificados[:30])

Municípios únicos em MG: 188
Não classificados: 0
Exemplos: []


### Exportação
Ao final desta etapa, são salvos três arquivos:

* a base completa com variáveis derivadas (DADOS_TRATADOS_MEDIAS_2022);
* o recorte de Minas Gerais com variável regional (DADOS_TRAT_MG_2022);
* o recorte específico de Caxambu (DADOS_TRAT_CAX_2022).

Esses arquivos servem de entrada para as etapas posteriores de consolidação longitudinal, dashboard e modelagem.

In [9]:
df_mg.to_parquet(DADOS_TRAT_MG_2022, index=False)  
df_caxambu.to_parquet(DADOS_TRAT_CAX_2022, index=False)
df.to_parquet(DADOS_TRATADOS_MEDIAS_2022, index=False)


print("✅ Salvo base completa com médias:", DADOS_TRATADOS_MEDIAS_2022, "| shape:", df.shape)
print("✅ Salvo recorte MG:", DADOS_TRAT_MG_2022, "| shape:", df_mg.shape)
print("✅ Salvo recorte Caxambu:", DADOS_TRAT_CAX_2022, "| shape:", df_caxambu.shape)

✅ Salvo base completa com médias: E:\ciencias_dados\projetos\projeto_enem_ml\dados\intermediarios\dados_tratados_2022_medias.parquet | shape: (3476105, 28)
✅ Salvo recorte MG: E:\ciencias_dados\projetos\projeto_enem_ml\dados\intermediarios\dados_tratados_MG_22.parquet | shape: (309781, 29)
✅ Salvo recorte Caxambu: E:\ciencias_dados\projetos\projeto_enem_ml\dados\intermediarios\dados_tratados_CAX_22.parquet | shape: (793, 28)
